# FastSurfer seg_only on IXI: which preprocessing step removes the hospital offset?

Ablation between the FreeSurfer conform baseline and Clinica t1-linear, same model and training:

| `--data` | N4 bias correction | skull stripping | registration to MNI |
|---|---|---|---|
| `conform` (done) | no | no | no |
| `fastsurfer_nu` | yes | no | no |
| `fastsurfer_brain` | yes | yes | no |
| `t1linear` (done) | yes | no | yes |

FastSurfer `--seg_only` (v2.6.1): conform, whole-brain segmentation, brain mask, N4. No registration, no surfaces, no FreeSurfer license needed.

**Runtime:** Runtime → Change runtime type → **T4 GPU**.

**Interruptions:** the outputs are written to Drive (`MyDrive/test-clinicadl/caps_fastsurfer_2mm`, ~1.5 GB in total), one subject at a time. After a disconnection, run cells 1-3 again, then cell 5: finished subjects are skipped. The same holds for the trainings (section 7): finished runs are skipped.

## 1. Drive and scripts

In [ ]:
import os, glob
import pandas as pd
from IPython.display import Image, display
from google.colab import drive
drive.mount('/content/drive')
os.environ['MPLBACKEND'] = 'Agg'   # scripts run with !python cannot use Colab's inline backend

DRIVE = '/content/drive/MyDrive/test-clinicadl'
WORK = '/content/test-clinicadl'
REPO = 'https://github.com/Ayoub-TAREK/cross-site-brain-age.git'
if os.path.exists(WORK) and not os.path.exists(f'{WORK}/.git'):
    !rm -rf {WORK}
if not os.path.exists(WORK):
    !git clone {REPO} {WORK}
else:
    !git -C {WORK} pull
os.makedirs(f'{WORK}/data/IXI/raw', exist_ok=True)
os.makedirs(f'{DRIVE}/outputs', exist_ok=True)
if not os.path.exists(f'{WORK}/outputs'):
    os.symlink(f'{DRIVE}/outputs', f'{WORK}/outputs')
FS_OUT = f'{DRIVE}/caps_fastsurfer_2mm'   # preprocessing outputs, kept on Drive
!git -C {WORK} log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Download IXI and convert to BIDS

≈ 4.5 GB, a few minutes; skipped if already there. Needed for the preprocessing only (not for section 7).

In [ ]:
!pip install -q xlrd
%cd {WORK}/data/IXI
URL = 'http://biomedic.doc.ic.ac.uk/brain-development/downloads/IXI'
if not os.path.exists('IXI.xls'):
    !curl -fL --retry 3 -o IXI.xls {URL}/IXI.xls
if len(glob.glob('raw/*.nii.gz')) < 500:
    !curl -fL --retry 3 -o IXI-T1.tar {URL}/IXI-T1.tar
    !tar -xf IXI-T1.tar -C raw && rm IXI-T1.tar
%cd {WORK}
!python scripts/ixi_to_bids.py

## 3. Install FastSurfer

Native install in its own environment (`/content/FastSurfer/.venv`, Python 3.12, PyTorch with CUDA); it cannot conflict with Colab's packages. About 5 min, to redo in every new session. The cell must end with `CUDA True` (`False` on a CPU runtime) and no error; an install that fails is redone at the next run of the cell.

In [ ]:
FS_HOME = '/content/FastSurfer'
FS_PY = f'{FS_HOME}/.venv/bin/python'
if not os.path.exists(f'{FS_HOME}/.installed'):   # written only once the install is checked
    !rm -rf {FS_HOME}
    !pip install -q uv
    !git -c advice.detachedHead=false clone -q --depth 1 --branch v2.6.1 https://github.com/Deep-MI/FastSurfer.git {FS_HOME}
    !cd {FS_HOME} && uv venv -q --python 3.12
    req = 'requirements.txt' if os.path.exists(f'{FS_HOME}/requirements.txt') else 'pyproject.toml'
    print('Installing the dependencies from', req)
    !cd {FS_HOME} && uv pip install --python {FS_PY} --torch-backend auto -r {req} 2>&1 | tail -n 5
    !cd {FS_HOME} && PYTHONPATH={FS_HOME} {FS_PY} FastSurferCNN/download_checkpoints.py --all 2>&1 | tail -n 3
    GPU = os.path.exists('/dev/nvidia0')   # a CPU runtime is possible for the second session (5 bis)
    !{FS_PY} -c "import torch, requests, nibabel; assert torch.cuda.is_available() or not {GPU}" && touch {FS_HOME}/.installed
!{FS_PY} -c "import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())"
assert os.path.exists(f'{FS_HOME}/.installed'), 'FastSurfer install failed: send the output above'

## 4. Timing test on 3 subjects

Processes the first 3 subjects not done yet (they count for the full run). Check the time per subject and the slices below: `nu` = conform + N4, `brain` = + skull stripping.

In [ ]:
%cd {WORK}
!python scripts/fastsurfer_preprocess.py --fastsurfer-home {FS_HOME} --out-dir {FS_OUT} --max-subjects 3

In [ ]:
%%writefile /content/show_fastsurfer.py
import sys, glob
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import nibabel as nib
import pandas as pd

out, png = sys.argv[1], sys.argv[2]
table = pd.read_csv(f'{out}/participants_sessions_age.tsv', sep='\t')
rows = []
for site in ['Guys', 'HH', 'IOP']:   # first finished subject of each hospital
    for _, r in table[table.site == site].iterrows():
        f = f'{out}/subjects/{r.participant_id}/{r.session_id}/fastsurfer/{r.participant_id}_{r.session_id}_desc-%s_res-2x2x2_T1w.nii.gz'
        if glob.glob(f % 'brain'):
            rows.append((site, r.participant_id, f))
            break
if not rows:
    sys.exit('No subject finished yet: nothing to show')
fig, axes = plt.subplots(len(rows), 6, figsize=(16, 2.8 * len(rows)), squeeze=False)
for ax_row, (site, sub, f) in zip(axes, rows):
    for k, desc in enumerate(['nu', 'brain']):
        data = nib.load(f % desc).get_fdata()
        x, y, z = (s // 2 for s in data.shape)
        for j, sl in enumerate([data[x], data[:, y], data[:, :, z]]):
            ax = ax_row[3 * k + j]
            ax.imshow(sl.T, cmap='gray', origin='lower')
            ax.axis('off')
        ax_row[3 * k].set_title(f'{site} {sub} - {desc}', loc='left', fontsize=9)
fig.tight_layout()
fig.savefig(png, dpi=80)

In [ ]:
import pandas as pd
!rm -f /content/fastsurfer_check.png && python /content/show_fastsurfer.py {FS_OUT} /content/fastsurfer_check.png
if os.path.exists('/content/fastsurfer_check.png'):
    display(Image('/content/fastsurfer_check.png'))
print(pd.read_csv(f'{FS_OUT}/fastsurfer_log.tsv', sep='\t').tail(5).to_string(index=False))

## 5. All subjects (resumable)

Stop it at any time (or lose the session): finished subjects are on Drive and skipped at the next launch. A failed subject is logged in `fastsurfer_log.tsv` (FastSurfer log in `failed/`) and retried at the next launch.

In [ ]:
%cd {WORK}
!python scripts/fastsurfer_preprocess.py --fastsurfer-home {FS_HOME} --out-dir {FS_OUT}

### 5 bis. Optional: a second session at the same time

To go about twice as fast, open this notebook in a **second Colab session** (same Drive), run cells 1-3 there, then this cell **instead of cell 5**. It starts from the last subject; the two sessions skip what the other one has finished and meet in the middle. Free Colab may refuse a second GPU at the same time: a CPU runtime also works, more slowly.

In [ ]:
%cd {WORK}
DEVICE = 'cuda' if os.path.exists('/dev/nvidia0') else 'cpu'
!python scripts/fastsurfer_preprocess.py --fastsurfer-home {FS_HOME} --out-dir {FS_OUT} --reverse --device {DEVICE}

## 6. Check the outputs

Count, time per subject, brain mask volume per hospital (a mask that failed on one hospital would show as outliers there), slices of one subject per hospital.

In [ ]:
import pandas as pd
n = len(glob.glob(f'{FS_OUT}/subjects/*/*/fastsurfer/*_desc-brain_res-2x2x2_T1w.nii.gz'))
print(n, 'subjects done (expected 563)')
log = pd.concat([pd.read_csv(f, sep='\t') for f in glob.glob(f'{FS_OUT}/fastsurfer_log*.tsv')])
log = log.drop_duplicates(['participant_id', 'session_id'], keep='last')
log = log.merge(pd.read_csv(f'{FS_OUT}/participants_sessions_age.tsv', sep='\t')[['participant_id', 'site']])
print(log.status.value_counts().to_string())
ok = log[log.status == 'ok']
print(ok.groupby('site').agg(n=('seconds', 'size'), minutes=('seconds', lambda s: round(s.mean() / 60, 1)),
                             mask_ml_mean=('mask_ml', 'mean'), mask_ml_min=('mask_ml', 'min'),
                             mask_ml_max=('mask_ml', 'max')).round(0).to_string())
!rm -f /content/fastsurfer_check.png && python /content/show_fastsurfer.py {FS_OUT} /content/fastsurfer_check.png
if os.path.exists('/content/fastsurfer_check.png'):
    display(Image('/content/fastsurfer_check.png'))

## 7. Training: 2 inputs × 3 held-out hospitals

Same setting as the conform baseline and v2 (seed 42, no augmentation, no scheduler, 50 epochs max, batch size 8), evaluated on the best validation epoch (`best-mae`, as all runs without scheduler). ~1 h per run on a T4. Needs cell 1 only (not IXI, not FastSurfer): the images are copied from Drive to the local disk. Finished runs are skipped.

In [ ]:
!pip install -q clinicadl==2.0.0
LOCAL = f'{WORK}/data/IXI/caps_fastsurfer_2mm'
n = len(glob.glob(f'{FS_OUT}/subjects/*/*/fastsurfer/*_desc-brain_res-2x2x2_T1w.nii.gz'))
assert n == 563, f'{n} / 563 subjects preprocessed: finish section 5 first'
if len(glob.glob(f'{LOCAL}/subjects/*/*/fastsurfer/*.nii.gz')) < 2 * 563:
    !rm -rf {LOCAL} && cp -r {FS_OUT} {LOCAL}
%cd {WORK}
!python scripts/train_ixi_site_shift.py --data fastsurfer_nu --check
!python scripts/train_ixi_site_shift.py --data fastsurfer_brain --check
for p in sorted(glob.glob(f'{WORK}/outputs/site_shift_test-IOP_fastsurfer_*/check_*.png')):
    print(os.path.basename(os.path.dirname(p)))
    display(Image(p))

In [ ]:
import shutil
for data in ['fastsurfer_nu', 'fastsurfer_brain']:
    for site in ['HH', 'Guys', 'IOP']:
        flags = f'--data {data} --test-site {site} --seed 42'
        run = f'{WORK}/outputs/site_shift_test-{site}_{data}'
        if os.path.exists(f'{run}/predictions_best-mae.tsv'):
            print('already done, skipped:', os.path.basename(run))
            continue
        for name in ['maps', 'split', 'train_sites.tsv', 'out_of_site.tsv']:   # unfinished earlier attempt
            path = os.path.join(run, name)
            if os.path.isdir(path):
                shutil.rmtree(path)
            elif os.path.exists(path):
                os.remove(path)
        print(f'\n===== {os.path.basename(run)} =====')
        !python scripts/train_ixi_site_shift.py {flags} --epochs 50 --batch-size 8 2>&1 | grep -E "seed :|Epoch [0-9]+ completed|stopped|Error|Traceback"
        !python scripts/predict_ages.py {flags} --checkpoint best-mae 2>&1 | grep -v -i warning | tail -n 12

## 8. Comparison with conform and t1-linear

Out-of-site MAE (in-site in parentheses) and out-of-site bias, best validation epoch, seed 42. Single runs: differences below ~2 years are not meaningful.

In [ ]:
rows = []
for data in ['conform', 'fastsurfer_nu', 'fastsurfer_brain', 't1linear']:
    for site in ['IOP', 'HH', 'Guys']:
        f = f'{WORK}/outputs/site_shift_test-{site}_{data}/predictions_best-mae.tsv'
        if not os.path.exists(f):
            continue
        df = pd.read_csv(f, sep='\t')
        ins, oos = df[df.group == 'insite'], df[df.group == 'outofsite']
        rows.append(dict(data=data, site=site,
                         mae=f'{oos.error.abs().mean():.1f} ({ins.error.abs().mean():.1f})',
                         bias=round(oos.error.mean(), 1), r=round(oos.AgeMRI.corr(oos.predicted), 2)))
res = pd.DataFrame(rows)
for col in ['mae', 'bias', 'r']:
    print(f'\n{col}')
    print(res.pivot(index='data', columns='site', values=col)
             .reindex(['conform', 'fastsurfer_nu', 'fastsurfer_brain', 't1linear']).to_string())